<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/LiDARtrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --force-reinstall numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have 

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter

import sys

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt


Mounted at /content/drive


In [3]:
!npx degit google-research-datasets/Objectron/objectron objectron

sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box

⠙⠹⠸⠼⠴Need to install the following packages:
degit@3.6.1
Ok to proceed? (y) y

⠙⠹⠸> cloned google-research-datasets/Objectron#HEAD to objectron
⠙

In [4]:
!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 6.769 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [5]:
from pyquaternion import Quaternion

sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder, get_ij, get_point_pillar

#import CenterPointHead

features(Δt, intensity) are kept in separate matrices, and pasted together for encoding

In [6]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)
#cloud_data = nusc.get("sample_data", cloud_sample["data"]["LIDAR_TOP"])

#cloud_root = os.path.join(nusc_root, cloud_data["filename"])

#cloud = LidarPointCloud.from_file(cloud_root)

#cloud_feats = cloud.points[3:,:].T

In [7]:
def load_sweeps(nusc, sample, N=10, min_distance = 1.0):

  pc, times = LidarPointCloud.from_file_multisweep(nusc=nusc, sample_rec = sample, chan = "LIDAR_TOP", ref_chan = "LIDAR_TOP", nsweeps = N, min_distance = min_distance)

  out = np.concatenate([pc.points, times], axis =0).T

  return out

In [8]:
def get_grid_attr(cloud, voxel_size = 1):

  min_x = np.min(cloud[:, 0:1])
  min_y = np.min(cloud[:, 1:2])
  min_z = np.min(cloud[:, 2:3])

  min_bound = np.array([min_x,min_y,min_z])

  max_x = np.max(cloud[:, 0:1])
  max_y = np.max(cloud[:, 1:2])
  max_z = np.max(cloud[:, 2:3])

  max_bound = np.array([max_x, max_y, max_z])

  pc_range = max_bound - min_bound

  grid_shape = np.ceil((max_bound - min_bound) / voxel_size).astype(int)

  return min_bound, max_bound, grid_shape, pc_range


In [9]:
def transform_cloud(cloud, pc_range =[-51.2,-51.2,51.2,51.2]):
  min_bound, max_bound, _, _= get_grid_attr(cloud)

  min_x, min_y = min_bound[0], min_bound[1]
  max_x, max_y = max_bound[0], max_bound[1]

  min = np.array([min_x, min_y])
  max = np.array([max_x, max_y])

  out = cloud.copy()
  out[:,:2] = out[:, :2] - min[None,:]

  out[:,:2] = out[:, :2] / (min-max)[None,:]
  out[:,:2] = out[:,:2] + 0.5

  out[:,:2] = out[:,:2]*2 * pc_range[2]

  return out

In [10]:
def get_pillars(cloud, voxel_size = 1, pillar_max = 100): #voxel_size is length of one side of voxel

  min_bound, max_bound, grid_shape = get_grid_attr(cloud, voxel_size=voxel_size)

  T = grid_shape[0]*grid_shape[1]

  N_counter = np.zeros(T)

  for k in range(cloud.shape[0]):
    point = cloud[k]

    p_index = get_point_pillar(point, min_bound, voxel_size, grid_shape[0])

    N_counter[p_index] += 1

  N = (np.max(N_counter)).astype(int)

  if pillar_max < N or pillar_max == None:
    N = pillar_max

  out = np.zeros((T, N, 5))

  mask = np.zeros((T), dtype = int)

  for m in range(cloud.shape[0]):
    point = cloud[m]

    p_index = get_point_pillar(point, min_bound, voxel_size, grid_shape[0])

    k = mask[p_index]

    if k < N:
      out[p_index][k] = point

      mask[p_index] += 1

  out_zeros = ~np.all(out == 0, axis=(1, 2))

  out = out[out_zeros,:,:]

  mask = mask[mask != 0] #mask returns how many non-padded points are in each pillar. Zero points in a pillar are assumed to be padded unless the mask indicates otherwise

  out = torch.from_numpy(out).float()
  mask = torch.from_numpy(mask).float()

  return out, mask


In [12]:
def process_boxes(nusc, sample, boxes, cloud, threshold = 1): #add function to include int labels that correspond to the categories on each box

  ego_pose = nusc.get("ego_pose", sample["data"]["LIDAR_TOP"])

  sd_rec = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
  cal_sensor = nusc.get('calibrated_sensor', sd_rec['calibrated_sensor_token'])

  out_boxes = []

  valid_boxes = []

  cloud_xyz = cloud[:,:3].T

  for box in boxes:
    category_name = box.name

    detection_name = category_to_detection_name(category_name)

    box.translate(-np.array(ego_pose['translation']))
    box.rotate(Quaternion(ego_pose['rotation']).inverse)

    box.translate(-np.array(cal_sensor['translation']))
    box.rotate(Quaternion(cal_sensor['rotation']).inverse)

    box.name = detection_name

    out_boxes.append(box)

  for box in out_boxes:
      indices = points_in_box(box, cloud_xyz)
      num_pts = np.sum(indices)

      if num_pts >= threshold:
        valid_boxes.append(box)

  return valid_boxes


In [ ]:
def get_boxes_max_min(boxes):

  x_min = boxes[0].center[0]
  y_min = boxes[0].center[1]
  x_max = boxes[0].center[0]
  y_max = boxes[0].center[1]

  for box in boxes:

    if box.center[0] < x_min:
      x_min = box.center[0]

    if box.center[1] < y_min:
      y_min = box.center[1]

    if box.center[0] > x_max:
      x_max = box.center[0]

    if box.center[1] > y_max:
      y_max = box.center[1]

  return x_min, y_min, x_max, y_max

def splat_gaussian(heatmap, center, sigma):

  r = heatmap.shape[0]

  x_axis = torch.arange(0, r, device=heatmap.device, dtype=torch.float32)
  y_axis = torch.arange(0, r, device=heatmap.device, dtype=torch.float32)
  y_grid, x_grid = torch.meshgrid(y_axis, x_axis, indexing='ij')

  distances_squared = torch.zeros((r,r), device=heatmap.device)

  distances_squared += (x_grid-center[0])**2 +(y_grid-center[1])**2

  gaussian = (-1*distances_squared)/(2*sigma**2)

  outmap = np.maximum(heatmap, gaussian)

  return outmap

def gaussian_radius(w, l, threshold = 0.1):

    a1, b1, c1 = 4, -2*(w+l), (1-threshold)*w*l
    r1 = (b1 - np.sqrt(b1**2 - 4*a1*c1)) / (2*a1)

    a2, b2, c2 = 4*threshold, 2*threshold*(w+l), -(1-threshold)*w*l
    r2 = (-b2 + np.sqrt(b2**2 - 4*a2*c2)) / (2*a2)

    a3, b3, c3 = 1, -(w+l), (1-1/threshold)*w*l
    r3 = (b3 + np.sqrt(b3**2 - 4*a3*c3)) / (2*a3)

    return max(min(r1, r2, r3),2)


def target_generation(gt_boxes, heatmap_resolution = 256, threshold = 0.1, k =10): #add index/mask tensor for regression targets

  box_ij = []
  box_wl = []

  x_min, y_min, x_max, y_max = get_boxes_max_min(gt_boxes)

  for box in gt_boxes:
    boxes.append(box)

    box_x = box.center[0]
    box_y = box.center[1]

    i = np.floor((box_x - x_min)/(x_max - x_min) * (heatmap_resolution - 1))
    j = np.floor((box_y - y_min)/(y_max - y_min) * (heatmap_resolution - 1))

    i = int(i)
    j = int(j)

    box_ij.append([i,j])
    box_wl.append([box.size[0], box.size[1]])

  sigmas = []

  for i in len(box_wl):
    w = box_wl[i][0]
    l = box_wl[i][1]

    sigma = gaussian_radius(w, l, threshold = threshold)

    sigmas.append(sigma)

  out_heatmaps = torch.zeros((heatmap_resolution, heatmap_resolution, k))

  for i in len(box_ij):
    label = gt_boxes[i].label
    label_map = out_heatmaps[:,:,label]

    out_heatmaps[:,:,label] = splat_gaussian(label_map, box_ij[i], sigmas[i])

  return out_heatmaps


In [15]:
out = load_sweeps(nusc, cloud_sample)

In [ ]:
pillars, mask = get_pillars(out)

In [ ]:
pillars.shape

torch.Size([2314, 100, 5])

In [16]:
boxes = nusc.get_boxes(cloud_sample["data"]['LIDAR_TOP'])

In [ ]:
tr_out = transform_cloud(out)

In [ ]:
get_grid_attr(tr_out)

(array([-51.2       , -51.2       ,  -3.41671157]),
 array([51.2       , 51.2       , 19.02801514]),
 array([103, 103,  23]),
 array([102.4       , 102.4       ,  22.44472671]))

box.corners() gives vertices

In [17]:
lidar_boxes = process_boxes(nusc, cloud_sample, boxes, out)

In [26]:
box_test = lidar_boxes[0]

In [27]:
box_test

label: nan, score: nan, xyz: [18.41, 59.52, 0.77], wlh: [0.62, 0.67, 1.64], rot axis: [0.01, -0.02, 1.00], ang(degrees): 179.02, ang(rad): 3.12, vel: nan, nan, nan, name: pedestrian, token: ef63a697930c4b20a6b9791f423351da